# Notebook 03c — Investigación: clientes españoles mal clasificados como internacionales

## Objetivo

Diagnosticar y corregir un caso identificado durante la construcción de `gold.cliente_360`: el cliente con mayor facturación de la cartera, El Corte Inglés (id_cliente 5728), aparece con país "Sin identificar" pese a ser una compañía claramente española. Este notebook investiga las causas del fallo, cuantifica su alcance sobre el conjunto de la cartera y propone una regla de clasificación complementaria a las tres reglas A, B y C documentadas en el notebook 03.

## Punto de partida

La clasificación NACIONAL/INTERNACIONAL implementada en `silver.dim_cliente` combina tres reglas de coherencia entre el código de provincia y el código postal. La aparición de ECI como "Sin identificar" sugiere que su codificación en el ERP no encaja en ninguna de las tres reglas, lo que abre la posibilidad de que existan más casos similares en la cartera.

## 1. Configuración y conexión a DuckDB

In [1]:
import duckdb
from pathlib import Path

RUTA_PROYECTO = Path.home() / "OneDrive" / "Documentos" / "TFG_Selmark"
RUTA_DUCKDB = RUTA_PROYECTO / "duckdb" / "selmark.duckdb"

con = duckdb.connect(str(RUTA_DUCKDB))
print(f"Conectado a: {RUTA_DUCKDB}")

Conectado a: C:\Users\lopec\OneDrive\Documentos\TFG_Selmark\duckdb\selmark.duckdb


## 2. Diagnóstico del caso El Corte Inglés

Se recupera el registro completo de El Corte Inglés en `silver.dim_cliente` para inspeccionar todos sus campos identificativos y geográficos, junto con su clasificación actual.

In [2]:
print("=" * 70)
print("DIAGNÓSTICO — Registro de ECI en silver.dim_cliente")
print("=" * 70)
eci = con.execute("""
    SELECT *
    FROM silver.dim_cliente
    WHERE id_cliente = '5728'
""").fetchdf()
print(eci.T)

DIAGNÓSTICO — Registro de ECI en silver.dim_cliente
                                              0
id_cliente                                 5728
nombre_cliente            EL CORTE INGLES, S.A.
nombre_comercial_cliente        EL CORTE INGLES
direccion_cliente                          null
localidad_cliente                          null
codigo_postal_original                     null
codigo_postal_norm                         null
codigo_provincia_cliente                   None
es_cliente_espanol                         True
tipo_mercado                           NACIONAL


## 3. Búsqueda de patrones similares

Se identifican los clientes que comparten el mismo patrón potencialmente problemático con ECI: clasificados como INTERNACIONAL pero con indicios de ser españoles (por su nombre, localidad o estructura del código postal).

In [3]:
print("=" * 70)
print("BÚSQUEDA — Clientes INTERNACIONAL con localidad española conocida")
print("=" * 70)

# Lista de localidades españolas comunes para hacer un primer filtro
LOCALIDADES_ESPANOLAS = (
    'MADRID', 'BARCELONA', 'VALENCIA', 'SEVILLA', 'ZARAGOZA',
    'MALAGA', 'MURCIA', 'PALMA', 'LAS PALMAS', 'BILBAO',
    'ALICANTE', 'CORDOBA', 'VALLADOLID', 'VIGO', 'GIJON',
    'A CORUÑA', 'GRANADA', 'OVIEDO', 'PAMPLONA', 'SANTANDER',
    'CASTELLON', 'BURGOS', 'SALAMANCA', 'LEON', 'CADIZ',
    'TARRAGONA', 'LERIDA', 'JAEN', 'OURENSE', 'GERONA',
    'LUGO', 'PONTEVEDRA', 'TOLEDO', 'BADAJOZ', 'HUELVA'
)

placeholders = ",".join([f"'{l}'" for l in LOCALIDADES_ESPANOLAS])
sospechosos = con.execute(f"""
    SELECT
        id_cliente,
        nombre_cliente,
        localidad_cliente,
        codigo_postal_norm,
        codigo_provincia_cliente,
        tipo_mercado
    FROM silver.dim_cliente
    WHERE tipo_mercado = 'INTERNACIONAL'
      AND UPPER(localidad_cliente) IN ({placeholders})
    ORDER BY nombre_cliente
""").fetchdf()
print(f"Clientes INTERNACIONAL con localidad española detectada: {len(sospechosos)}")
print(sospechosos.to_string(index=False))

BÚSQUEDA — Clientes INTERNACIONAL con localidad española conocida
Clientes INTERNACIONAL con localidad española detectada: 0
Empty DataFrame
Columns: [id_cliente, nombre_cliente, localidad_cliente, codigo_postal_norm, codigo_provincia_cliente, tipo_mercado]
Index: []


## 4. Análisis de los códigos de provincia en los casos sospechosos

Se examinan los códigos de provincia presentes en los clientes detectados, para identificar el patrón de codificación que actualmente está quedando fuera de las reglas A, B y C.

In [4]:
print("=" * 70)
print("ANÁLISIS — Códigos de provincia en clientes sospechosos")
print("=" * 70)
patrones = con.execute(f"""
    SELECT
        codigo_provincia_cliente,
        codigo_postal_norm,
        COUNT(*) AS num_clientes,
        STRING_AGG(DISTINCT localidad_cliente, ', ' ORDER BY localidad_cliente) AS localidades
    FROM silver.dim_cliente
    WHERE tipo_mercado = 'INTERNACIONAL'
      AND UPPER(localidad_cliente) IN ({placeholders})
    GROUP BY codigo_provincia_cliente, codigo_postal_norm
    ORDER BY num_clientes DESC
""").fetchdf()
print(patrones.to_string(index=False))

ANÁLISIS — Códigos de provincia en clientes sospechosos
Empty DataFrame
Columns: [codigo_provincia_cliente, codigo_postal_norm, num_clientes, localidades]
Index: []


## 5. Conclusiones preliminares

Esta sección sintetizará los hallazgos del diagnóstico una vez ejecutadas las celdas anteriores. La interpretación de los patrones detectados orientará el diseño de la regla correctiva, que se implementará en una segunda fase del notebook o, en caso de que el alcance del problema lo justifique, mediante una modificación directa del notebook 03.